In [ ]:
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
import os
import numpy as np
pd.set_option('display.max_columns', None)

In [46]:
FECHA_PERIODO= "2025-08-20"

In [36]:
df_desgravamen= pd.read_excel('C:/data/410 - Errores - muestra.xlsx', sheet_name='Sheet1', 
                              dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str, 
                                     'IDELOTE': str, 'CODIGO ERROR': str, 
                                     'SUMA ASEGURADA':str, 'TASA': str, 
                                     'TASA RECARGO': str, 'PRIMABRUTACAN':str, 'PRIMANETACAN': str})

In [37]:
df_desgravamen.columns = (df_desgravamen.columns.str.strip()
                          .str.upper()  # opcional: todo en mayúsculas
                          .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar estapacios por _
                        )

In [38]:
df_desgravamen.drop(['IDELOTE','IDEDET_1','DESCRIPCION_ERROR'], axis=1, inplace=True)
df_desgravamen= df_desgravamen.rename(columns={'CODIGO_ERROR': 'IDEERROR'})

In [39]:
df_desgravamen= df_desgravamen.fillna({'LOTES_ANTERIORES':'', 'MONEDA': 'SIN DATO', 'ORIGEN_ERROR': 'SIN DATO', 'TIPDOCUMENTO':'SIN DATO'})
df_desgravamen.loc[df_desgravamen['MONEDA'] == 'nan', 'MONEDA'] = 'SIN DATO'
df_desgravamen.loc[df_desgravamen['ORIGEN_ERROR'] == 'nan', 'ORIGEN_ERROR'] = 'SIN DATO'

In [40]:
df_desgravamen['SUMA_ASEGURADA'] = df_desgravamen['SUMA_ASEGURADA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA'] = df_desgravamen['TASA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA_RECARGO'] = df_desgravamen['TASA_RECARGO'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMABRUTACAN'] = df_desgravamen['PRIMABRUTACAN'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMANETACAN'] = df_desgravamen['PRIMANETACAN'].str.replace(',', '.', regex=False)

In [41]:
def limpiar_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

In [35]:
df_desgravamen['TASA_RECARGO'].value_counts()

TASA_RECARGO
0    999
Name: count, dtype: int64

In [42]:
df_desgravamen["LOTES_ANTERIORES"] = df_desgravamen["LOTES_ANTERIORES"].astype(str)
df_desgravamen['FECHA_CARGA'] = pd.to_datetime(df_desgravamen['FECHA_CARGA'], format="%d/%m/%Y %I:%M:%S %p", errors='coerce')
df_desgravamen['CODIGO_PRODUCTO'] = df_desgravamen['CODIGO_PRODUCTO'].fillna(0).astype('int')
df_desgravamen["PRODUCTO"] = df_desgravamen["PRODUCTO"].astype(str)
df_desgravamen['CODIGO_PLAN'] = df_desgravamen['CODIGO_PLAN'].fillna(0).astype('int')
df_desgravamen["NOMBRE_DE_PLAN"] = df_desgravamen["NOMBRE_DE_PLAN"].astype(str)
df_desgravamen["COD_DE_CERTIFICADO"] = df_desgravamen["COD_DE_CERTIFICADO"].astype(str)
df_desgravamen['FECINI_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECINI_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FECFIN_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECFIN_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen["TIPO_MOVIMIENTO"] = df_desgravamen["TIPO_MOVIMIENTO"].astype(str)
df_desgravamen['FEC__INICIO'] = pd.to_datetime(df_desgravamen['FEC__INICIO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FEC__FIN'] = pd.to_datetime(df_desgravamen['FEC__FIN'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['MONEDA'] = df_desgravamen['MONEDA'].astype(str)
df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce").astype('float64')
df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce").astype('float64')
df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce").astype('float64')
df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce").astype('float64')
df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce").astype('float64')
df_desgravamen['NOMCOMPLETO'] = df_desgravamen['NOMCOMPLETO'].astype(str)
df_desgravamen['APEPATERNO'] = df_desgravamen['APEPATERNO'].astype(str)
df_desgravamen['APEMATERNO'] = df_desgravamen['APEMATERNO'].astype(str)
df_desgravamen["FECNACIMIENTO"] = limpiar_fecha(df_desgravamen["FECNACIMIENTO"])
df_desgravamen["TIPDOCUMENTO"] = df_desgravamen["TIPDOCUMENTO"].astype(str)
df_desgravamen['NUMDOCUMENTO'] = df_desgravamen['NUMDOCUMENTO'].astype(str)
df_desgravamen['NOMBRE_DE_ARCHIVO'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].astype(str)
df_desgravamen['LINEA_TRAMA'] = df_desgravamen['LINEA_TRAMA'].astype(str)
df_desgravamen['ORIGEN_ERROR'] = df_desgravamen['ORIGEN_ERROR'].astype(str)
df_desgravamen['IDEERROR'] = df_desgravamen['IDEERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")

In [45]:
#### FUNCION PARA EXTRAER LA FECHA DEL NOMBRE DE ARCHIVO
def extraer_fecha(fecha):
    try:
        fecha_str = fecha.split("_")[2]
        return datetime.strptime(fecha_str, "%Y%m%d").date()
    except Exception:
        return None  # En caso de error

#### FUNCION PARA CLASIFICAR EL REGISTRO
def clasificar(row):
    # Si lote no es nulo
    if str(row['LOTES_ANTERIORES']).strip() != "":
        return 'Regularizacion no Exitosa'

    # Si último número del nombre de archivo > 100
    try:
        ultimo_numero = int(row['NOMBRE_DE_ARCHIVO'].split("_")[-1][:-4])
        if ultimo_numero > 100:
            return 'Regularizacion no Exitosa'
    except:
        pass

    # Si ninguna condición se cumple
    return 'Mes corriente'

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return

In [ ]:
df_desgravamen= df_desgravamen[df_desgravamen['PRIMABRUTACAN']<30000]   #Eliminar las filas con valores atipicos muy altos
# Crear nuevas columnas cextrayendo la fecha del nombre de archivo
df_desgravamen['FECHA_TRAMA'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].apply(extraer_fecha)
df_desgravamen['FECHA_TRAMA']= pd.to_datetime(df_desgravamen['FECHA_TRAMA'], errors='coerce')
# Columna para diferenciar el periodo de reporte de errores
df_desgravamen['FECHA_PERIODO']= pd.to_datetime(FECHA_PERIODO)
# Crear columna evaluando condiciones en el contenido de otras columnas
df_desgravamen['TIPO_ERROR']= df_desgravamen.apply(clasificar, axis=1)

In [48]:
df_desgravamen.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,IDEERROR,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR
0,40856,,2015-03-16,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110125254000126298,NaT,NaT,65909536,Renovacion,2014-09-19,2015-01-09,SOL,4.24,0.0,0.0,5.0,4.85,PAOLA,PEREZ,PINEDO,1983-06-16,2,41961269,20100130204_0158001_20150310_002.TXT,918001101252540001262980011012521500140148501P...,Error canal,1153,2015-03-10,2025-08-20,Mes corriente
1,41817,,2015-04-01,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110133684000194238,NaT,NaT,69920930,Exclusion,NaT,2015-02-24,SOL,0.00,0.0,0.0,0.0,0.00,ERIKA SHEILA,MENESES,CUZCANO,1983-07-09,2,41957497,20100130204_0158001_20150331_006.TXT,918001101336840001942380011013360500143789001P...,Error canal,1150,2015-03-31,2025-08-20,Mes corriente
2,41817,,2015-04-01,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110785334000394305,NaT,NaT,69921506,Exclusion,NaT,2015-02-19,SOL,0.00,0.0,0.0,0.0,0.00,MIGUEL ANGEL,ARRIARAN,ESCRIBA,1983-03-09,2,43723290,20100130204_0158001_20150331_006.TXT,918001107853340003943050011078533500198853101P...,Error canal,1150,2015-03-31,2025-08-20,Mes corriente


In [49]:
df_desgravamen.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   NRO_LOTE                 999 non-null    int64         
 1   LOTES_ANTERIORES         999 non-null    object        
 2   FECHA_CARGA              999 non-null    datetime64[ns]
 3   CODIGO_PRODUCTO          999 non-null    int64         
 4   PRODUCTO                 999 non-null    object        
 5   CODIGO_PLAN              999 non-null    int64         
 6   NOMBRE_DE_PLAN           999 non-null    object        
 7   COD_DE_CERTIFICADO       999 non-null    object        
 8   FECINI_ALTA_CERTIFICADO  723 non-null    datetime64[ns]
 9   FECFIN_ALTA_CERTIFICADO  723 non-null    datetime64[ns]
 10  IDEDET                   999 non-null    int64         
 11  TIPO_MOVIMIENTO          999 non-null    object        
 12  FEC__INICIO              1 non-null 